你可以從 [Bookshop.org](https://bookshop.org/a/98697/9781098155438) 和 [Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325) 訂購 *Think Python 3e* 的紙本和電子書版本。

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

感謝：圖片下載自 [Lorem Picsum](https://picsum.photos/)，這是一個提供佔位圖片的服務。
此名稱來自「lorem ipsum」，是佔位文字的通稱。

In [2]:
# This cell downloads an archive file that contains the the files we'll
# use for the examples in this chapter.

download('https://github.com/AllenDowney/ThinkPython/raw/v3/photos.zip');

In [3]:
# WARNING: This cell removes the photos/ directory if it already exists.
# Any files already in the photos/ directory will be deleted.

# !rm -rf photos/

In [4]:
!unzip -o photos.zip

Archive:  photos.zip
  inflating: photos/notes.txt        
  inflating: photos/mar-2023/photo2.jpg  
  inflating: photos/mar-2023/photo1.jpg  
  inflating: photos/jan-2023/photo3.jpg  
  inflating: photos/jan-2023/photo2.jpg  
  inflating: photos/jan-2023/photo1.jpg  
  inflating: photos/feb-2023/photo2.jpg  
  inflating: photos/feb-2023/photo1.jpg  


# 檔案和資料庫

到目前為止，我們看到的大多數程式都是**暫時性的**，它們運行一小段時間並產生輸出，但當它們結束時，資料就會消失。每次運行暫時性程式時，它都會從頭開始。

其他程式是**持久性的**：它們運行很長時間（或一直運行）；它們將至少一部分資料保存在長期儲存中；如果它們關閉並重新啟動，它們會從中斷的地方繼續。

程式維護資料的一個簡單方法是讀取和寫入文字檔案。
更多功能的替代方案是將資料儲存在資料庫中。
資料庫是專門的檔案，可以比文字檔案更有效地讀取和寫入，並提供額外的功能。

在本章中，我們將編寫讀取和寫入文字檔案和資料庫的程式，作為練習，你將編寫一個在照片集合中搜尋重複檔案的程式。
但在你可以處理檔案之前，你必須找到它，所以我們將從檔案名稱、路徑和目錄開始。

## 檔案名稱和路徑

檔案被組織在**目錄**中，也稱為「資料夾」。
每個運行的程式都有一個**當前工作目錄**，這是大多數操作的預設目錄。
例如，當你開啟一個檔案時，Python 會在當前工作目錄中尋找它。

`os` 模組提供了處理檔案和目錄的功能（「os」代表「作業系統」）。
它提供了一個名為 `getcwd` 的函數，用來取得當前工作目錄的名稱。

In [5]:
# this cell replaces `os.cwd` with a function that returns a fake path

import os

def getcwd():
    return "/home/dinsdale"

os.getcwd = getcwd

In [6]:
import os

os.getcwd()

'/home/dinsdale'

此範例中的結果是名為 `dinsdale` 的使用者的家目錄。
像 `'/home/dinsdale'` 這樣識別檔案或目錄的字串稱為**路徑**。

像 `'memo.txt'` 這樣的簡單檔案名稱也被視為路徑，但它是一個**相對路徑**，因為它指定相對於當前目錄的檔案名稱。
在此範例中，當前目錄是 `/home/dinsdale`，所以 `'memo.txt'` 等同於完整路徑 `'/home/dinsdale/memo.txt'`。

以 `/` 開頭的路徑不依賴於當前目錄——它稱為**絕對路徑**。
要尋找檔案的絕對路徑，你可以使用 `abspath`。

In [7]:
os.path.abspath('memo.txt')

'/home/dinsdale/memo.txt'

`os` 模組提供了其他處理檔案名稱和路徑的功能。
`listdir` 返回給定目錄內容的列表，包括檔案和其他目錄。
以下是一個列出名為 `photos` 目錄內容的範例。

In [8]:
os.listdir('photos')

['digests.dat',
 'digests.dir',
 'notes.txt',
 'new_notes.txt',
 'mar-2023',
 'digests.bak',
 'jan-2023',
 'feb-2023']

此目錄包含一個名為 `notes.txt` 的文字檔案和三個目錄。
這些目錄包含 JPEG 格式的圖片檔案。

In [9]:
os.listdir('photos/jan-2023')

['photo3.jpg', 'photo2.jpg', 'photo1.jpg']

要檢查檔案或目錄是否存在，我們可以使用 `os.path.exists`。

In [10]:
os.path.exists('photos')

True

In [11]:
os.path.exists('photos/apr-2023')

False

要檢查路徑是否指向檔案或目錄，我們可以使用 `isdir`，當路徑指向目錄時返回 `True`。

In [12]:
os.path.isdir('photos')

True

而 `isfile` 當路徑指向檔案時返回 `True`。

In [13]:
os.path.isfile('photos/notes.txt')

True

處理路徑的一個挑戰是它們在不同的作業系統上看起來不同。
在 macOS 和 Linux 等 UNIX 系統上，路徑中的目錄和檔案名稱以正斜線 `/` 分隔。
Windows 使用反斜線 `\`。
所以，如果你在 Windows 上運行這些範例，你會在路徑中看到反斜線，而且你必須將範例中的正斜線替換掉。

或者，要編寫在兩個系統上都能運行的程式碼，你可以使用 `os.path.join`，它會根據你使用的作業系統，使用正斜線或反斜線將目錄和檔案名稱連接成路徑。

In [14]:
os.path.join('photos', 'jan-2023', 'photo1.jpg')

'photos/jan-2023/photo1.jpg'

在本章稍後，我們將使用這些函數來搜尋一組目錄並找到所有的圖片檔案。

## f-字串

程式儲存資料的一種方法是將其寫入文字檔案。
例如，假設你是一個駱駝觀察者，你想記錄在觀察期間看到的駱駝數量。
假設在一年半的時間裡，你發現了 `23` 隻駱駝。
你的駱駝觀察記錄本中的資料可能是這樣的。

In [15]:
num_years = 1.5
num_camels = 23

要將此資料寫入檔案，你可以使用我們在第8章中看到的 `write` 方法。
`write` 的參數必須是字串，所以如果我們想將其他值放入檔案中，我們必須將它們轉換為字串。
最簡單的方法是使用內建函數 `str`。

以下是它的樣子：

In [16]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(str(num_years))
writer.write(str(num_camels))
writer.close()

這樣做有效，但 `write` 不會添加空格或換行符，除非你明確包含它。
如果我們讀回檔案，我們會看到兩個數字連在一起。

In [17]:
open('camel-spotting-book.txt').read()

'1.523'

至少，我們應該在數字之間添加空白字元。
而且既然我們要這樣做，讓我們也添加一些說明文字。

要編寫字串和其他值的組合，我們可以使用 **f-字串**，這是一個在開頭引號前有字母 `f` 的字串，並在大括號中包含一個或多個 Python 表達式。
以下 f-字串包含一個表達式，它是一個變數名稱。

In [18]:
f'I have spotted {num_camels} camels'

'I have spotted 23 camels'

結果是一個字串，其中表達式已被評估並替換為結果。
可以有多個表達式。

In [19]:
f'In {num_years} years I have spotted {num_camels} camels'

'In 1.5 years I have spotted 23 camels'

表達式可以包含運算子和函數呼叫。

In [20]:
line = f'In {round(num_years * 12)} months I have spotted {num_camels} camels'
line

'In 18 months I have spotted 23 camels'

所以我們可以像這樣將資料寫入文字檔案。

In [21]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(f'Years of observation: {num_years}\n')
writer.write(f'Camels spotted: {num_camels}\n')
writer.close()

兩個 f-字串都以序列 `\n` 結尾，這會添加一個換行字元。

我們可以像這樣讀回檔案：

In [22]:
data = open('camel-spotting-book.txt').read()
print(data)

Years of observation: 1.5
Camels spotted: 23



在 f-字串中，大括號中的表達式會被轉換為字串，所以你可以包含列表、字典和其他類型。

In [23]:
t = [1, 2, 3]
d = {'one': 1}
f'Here is a list {t} and a dictionary {d}'

"Here is a list [1, 2, 3] and a dictionary {'one': 1}"

如果 f-字串包含無效的表達式，結果會是錯誤。

In [24]:
%%expect TypeError

f'This is not a valid expression {t + 2}'

TypeError: can only concatenate list (not "int") to list

## YAML

程式讀取和寫入檔案的原因之一是儲存**配置資料**，這是指定程式應該做什麼以及如何做的資訊。

例如，在搜尋重複照片的程式中，我們可能有一個名為 `config` 的字典，其中包含要搜尋的目錄名稱、應該儲存結果的另一個目錄名稱，以及應該用來識別圖片檔案的檔案副檔名列表。

它可能看起來像這樣：

In [25]:
config = {
    'photo_dir': 'photos',
    'data_dir': 'photo_info',
    'extensions': ['jpg', 'jpeg'],
}

要將此資料寫入文字檔案，我們可以使用 f-字串，如前一節所述。但使用名為 `yaml` 的模組更容易，它就是為這類事情而設計的。

`yaml` 模組提供函數來處理 YAML 檔案，這些是格式化為易於人類*和*程式讀取和寫入的文字檔案。

以下是使用 `dump` 函數將 `config` 字典寫入 YAML 檔案的範例。

In [26]:
# this cell installs the pyyaml package, which provides the yaml module

try:
    import yaml
except ImportError:
    !pip install pyyaml

In [27]:
import yaml

config_filename = 'config.yaml'
writer = open(config_filename, 'w')
yaml.dump(config, writer)
writer.close()

如果我們讀回檔案的內容，我們可以看到 YAML 格式是什麼樣子。

In [28]:
readback = open(config_filename).read()
print(readback)

data_dir: photo_info
extensions:
- jpg
- jpeg
photo_dir: photos



現在，我們可以使用 `safe_load` 來讀回 YAML 檔案。

In [29]:
reader = open(config_filename)
config_readback = yaml.safe_load(reader)
config_readback

{'data_dir': 'photo_info',
 'extensions': ['jpg', 'jpeg'],
 'photo_dir': 'photos'}

結果是一個新的字典，包含與原始字典相同的資訊，但它不是同一個字典。

In [30]:
config is config_readback

False

將像字典這樣的物件轉換為字串稱為**序列化**。
將字串轉換回物件稱為**反序列化**。
如果你序列化然後反序列化一個物件，結果應該等同於原始物件。

## Shelve

到目前為止，我們一直在讀取和寫入文字檔案——現在讓我們考慮資料庫。
**資料庫**是一個為儲存資料而組織的檔案。
有些資料庫像表格一樣組織，有行和列的資訊。
其他資料庫像字典一樣組織，從鍵映射到值；它們有時被稱為**鍵值儲存**。

`shelve` 模組提供了創建和更新名為「shelf」的鍵值儲存的功能。
作為範例，我們將創建一個 shelf 來包含 `photos` 目錄中圖片的標題。
我們將使用 `config` 字典來取得應該放置 shelf 的目錄名稱。

In [31]:
config['data_dir']

'photo_info'

我們可以使用 `os.makedirs` 來創建此目錄（如果它不存在的話）。

In [32]:
os.makedirs(config['data_dir'], exist_ok=True)

以及使用 `os.path.join` 來建立包含目錄名稱和 shelf 檔案名稱 `captions` 的路徑。

In [33]:
db_file = os.path.join(config['data_dir'], 'captions')
db_file

'photo_info/captions'

現在我們可以使用 `shelve.open` 來開啟 shelf 檔案。
參數 `c` 表示如果需要的話應該創建檔案。

In [34]:
import shelve

db = shelve.open(db_file, 'c')
db

返回值正式是一個 `DbfilenameShelf` 物件，比較隨意的稱法是 shelf 物件。

shelf 物件在很多方面表現得像字典。
例如，我們可以使用方括號運算子來添加一個項目，它是從鍵到值的映射。

In [35]:
key = 'jan-2023/photo1.jpg' 
db[key] = 'Cat nose'

在此範例中，鍵是圖片檔案的路徑，值是描述圖片的字串。

我們也使用方括號運算子來查找鍵並取得對應的值。

In [36]:
value = db[key]
value

'Cat nose'

如果你對現有的鍵進行另一次賦值，`shelve` 會替換舊值。

In [37]:
db[key] = 'Close up view of a cat nose'
db[key]

'Close up view of a cat nose'

一些字典方法，如 `keys`、`values` 和 `items`，也能與 shelf 物件一起使用。

In [38]:
list(db.keys())

['jan-2023/photo1.jpg']

In [39]:
list(db.values())

['Close up view of a cat nose']

我們可以使用 `in` 運算子來檢查鍵是否出現在 shelf 中。

In [40]:
key in db

True

我們可以使用 `for` 陳述式來遍歷鍵。

In [41]:
for key in db:
    print(key, ':', db[key])

jan-2023/photo1.jpg : Close up view of a cat nose


與其他檔案一樣，當你完成時應該關閉資料庫。

In [42]:
db.close()

現在如果我們列出資料目錄的內容，我們會看到兩個檔案。

In [43]:
# When you open a shelve file, a backup file is created that has the suffix `.bak`.
# If you run this notebook more than once, you might see that file left behind.
# This cell removes it so the output shown in the book is correct.

!rm -f photo_info/captions.bak

In [44]:
os.listdir(config['data_dir'])

['captions.dir', 'captions.dat']

`captions.dat` 包含我們剛才儲存的資料。
`captions.dir` 包含有關資料庫組織的資訊，這使得存取更有效率。
後綴 `dir` 代表「directory」，但與我們一直在處理的包含檔案的目錄無關。

## 儲存資料結構

在前面的範例中，shelf 中的鍵和值都是字串。
但我們也可以使用 shelf 來包含像列表和字典這樣的資料結構。

作為範例，讓我們重新審視第11章練習中的變位詞範例。
回想一下，我們製作了一個字典，從排序的字母字串映射到可以用這些字母拼寫的詞列表。
例如，鍵 `'opst'` 映射到列表 `['opts', 'post', 'pots', 'spot', 'stop', 'tops']`。

我們將使用以下函數來排序詞中的字母。

In [45]:
def sort_word(word):
    return ''.join(sorted(word))

以下是一個範例。

In [46]:
word = 'pots'
key = sort_word(word)
key

'opst'

現在讓我們開啟一個名為 `anagram_map` 的 shelf。
參數 `'n'` 表示我們應該總是創建一個新的空 shelf，即使已經存在一個。

In [47]:
db = shelve.open('anagram_map', 'n')

現在我們可以像這樣向 shelf 添加項目。

In [48]:
db[key] = [word]
db[key]

['pots']

在此項目中，鍵是字串，值是字串列表。

現在假設我們找到另一個包含相同字母的詞，如 `tops`

In [49]:
word = 'tops'
key = sort_word(word)
key

'opst'

鍵與前面的範例相同，所以我們想要將第二個詞附加到相同的字串列表中。
如果 `db` 是字典，我們會這樣做。

In [50]:
db[key].append(word)          # INCORRECT

但如果我們運行那個，然後在 shelf 中查找鍵，看起來它沒有被更新。

In [51]:
db[key]

['pots']

問題如下：當我們查找鍵時，我們得到一個字串列表，但如果我們修改字串列表，它不會影響 shelf。
如果我們想要更新 shelf，我們必須讀取舊值，更新它，然後將新值寫回 shelf。

In [52]:
anagram_list = db[key]
anagram_list.append(word)
db[key] = anagram_list

現在 shelf 中的值已更新。

In [53]:
db[key]

['pots', 'tops']

作為練習，你可以通過讀取詞彙列表並將所有變位詞儲存在 shelf 中來完成此範例。

In [54]:
db.close()

## 檢查等效檔案

現在讓我們回到本章的目標：搜尋包含相同資料的不同檔案。
檢查的一種方法是讀取兩個檔案的內容並比較。

如果檔案包含圖片，我們必須以模式 `'rb'` 開啟它們，其中 `'r'` 表示我們想要讀取內容，`'b'` 表示**二進位模式**。
在二進位模式中，內容不被解釋為文字——它們被視為位元組序列。

以下是開啟和讀取圖片檔案的範例。

In [55]:
path1 = 'photos/jan-2023/photo1.jpg'
data1 = open(path1, 'rb').read()
type(data1)

bytes

`read` 的結果是一個 `bytes` 物件——如其名稱所示，它包含位元組序列。

一般來說，圖片檔案的內容不是人類可讀的。
但如果我們從第二個檔案讀取內容，我們可以使用 `==` 運算子來比較。

In [56]:
path2 = 'photos/jan-2023/photo2.jpg'
data2 = open(path2, 'rb').read()
data1 == data2

False

這兩個檔案不相等。

讓我們將到目前為止的內容封裝在一個函數中。

In [57]:
def same_contents(path1, path2):
    data1 = open(path1, 'rb').read()
    data2 = open(path2, 'rb').read()
    return data1 == data2

如果我們只有兩個檔案，此函數是一個好選擇。
但假設我們有大量檔案，我們想知道其中任何兩個是否包含相同的資料。
比較每對檔案是低效的。

另一種方法是使用**雜湊函數**，它接受檔案的內容並計算**摘要**，通常是一個大整數。
如果兩個檔案包含相同的資料，它們將具有相同的摘要。
如果兩個檔案不同，它們*幾乎總是*會有不同的摘要。

`hashlib` 模組提供了幾個雜湊函數——我們將使用的是名為 `md5`。
我們將開始使用 `hashlib.md5` 來創建一個 `HASH` 物件。

In [58]:
import hashlib

md5_hash = hashlib.md5()
type(md5_hash)

_hashlib.HASH

`HASH` 物件提供一個 `update` 方法，它將檔案的內容作為參數。

In [59]:
md5_hash.update(data1)

現在我們可以使用 `hexdigest` 來取得作為十六進位數字字串的摘要，它們代表基數16中的整數。

In [60]:
digest = md5_hash.hexdigest()
digest

'aa1d2fc25b7ae247b2931f5a0882fa37'

以下函數封裝了這些步驟。

In [61]:
def md5_digest(filename):
    data = open(filename, 'rb').read()
    md5_hash = hashlib.md5()
    md5_hash.update(data)
    digest = md5_hash.hexdigest()
    return digest

如果我們雜湊不同檔案的內容，我們可以確認我們得到不同的摘要。

In [62]:
filename2 = 'photos/feb-2023/photo2.jpg'
md5_digest(filename2)

'6a501b11b01f89af9c3f6591d7f02c49'

現在我們幾乎有了尋找等效檔案所需的一切。
最後一步是搜尋目錄並找到所有的圖片檔案。

## 遍歷目錄

以下函數以要搜尋的目錄作為參數。
它使用 `listdir` 來遍歷目錄的內容。
當它找到檔案時，它會印出其完整路徑。
當它找到目錄時，它會遞迴呼叫自己來搜尋子目錄。

In [63]:
def walk(dirname):
    for name in os.listdir(dirname):
        path = os.path.join(dirname, name)

        if os.path.isfile(path):
            print(path)
        elif os.path.isdir(path):
            walk(path)

我們可以像這樣使用它：

In [64]:
walk('photos')

photos/digests.dat
photos/digests.dir
photos/notes.txt
photos/new_notes.txt
photos/mar-2023/photo2.jpg
photos/mar-2023/photo1.jpg
photos/digests.bak
photos/jan-2023/photo3.jpg
photos/jan-2023/photo2.jpg
photos/jan-2023/photo1.jpg
photos/feb-2023/photo2.jpg
photos/feb-2023/photo1.jpg


結果的順序取決於作業系統的細節。

## 除錯

當你讀取和寫入檔案時，你可能會遇到空白字元的問題。
這些錯誤可能很難除錯，因為空白字元通常是不可見的。
例如，以下是一個包含空格、以序列 `\t` 表示的tab，以及以序列 `\n` 表示的換行的字串。
當我們印出它時，我們看不到空白字元。

In [65]:
s = '1 2\t 3\n 4'
print(s)

1 2	 3
 4


內建函數 `repr` 可以幫助。它接受任何物件作為參數並返回物件的字串表示。
對於字串，它用反斜線序列表示空白字元。

In [66]:
print(repr(s))

'1 2\t 3\n 4'


這對除錯很有幫助。

你可能遇到的另一個問題是不同系統使用不同的字元來表示行結束。一些系統使用換行，表示為 `\n`。其他系統使用回車字元，表示為 `\r`。
有些使用兩者。如果你在不同系統之間移動檔案，這些
不一致可能會造成問題。

檔案名稱大小寫是如果你使用不同作業系統可能遇到的另一個問題。
在 macOS 和 UNIX 中，檔案名稱可以包含小寫和大寫字母、數字和大多數符號。
但許多 Windows 應用程式忽略小寫和大寫字母之間的差異，而且一些在 macOS 和 UNIX 中允許的符號在 Windows 中不被允許。

## 詞彙表

**暫時性的：**
暫時性程式通常運行一小段時間，當它結束時，其資料會遺失。

**持久性的：**
持久性程式無限期運行，並將至少一部分資料保存在永久儲存中。

**目錄：**
檔案和其他目錄的集合。

**當前工作目錄：**
程式使用的預設目錄，除非指定另一個目錄。

**路徑：**
指定目錄序列的字串，通常指向檔案。

**相對路徑：**
從當前工作目錄或某個其他指定目錄開始的路徑。

**絕對路徑：**
不依賴於當前目錄的路徑。

**f-字串：**
在開頭引號前有字母 `f` 的字串，並在大括號中包含一個或多個表達式。

**配置資料：**
通常儲存在檔案中的資料，指定程式應該做什麼以及如何做。

**序列化：**
將物件轉換為字串。

**反序列化：**
將字串轉換為物件。

**資料庫：**
其內容經過組織以有效執行某些操作的檔案。

**鍵值儲存：**
其內容像字典一樣組織的資料庫，具有對應值的鍵。

**二進位模式：**
開啟檔案的方式，使內容被解釋為位元組序列而不是字元序列。

**雜湊函數：**
接受物件並計算整數的函數，有時稱為摘要。

**摘要：**
雜湊函數的結果，特別是當它用於檢查兩個物件是否相同時。

## 練習

In [67]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

Exception reporting mode: Verbose


### 詢問虛擬助理

在本章中出現了幾個我沒有詳細解釋的主題。
以下是一些你可以詢問虛擬助理以獲得更多資訊的問題。

* 「暫時性程式和持久性程式有什麼區別？」

* 「持久性程式有哪些例子？」

* 「相對路徑和絕對路徑有什麼區別？」

* 「為什麼 `yaml` 模組有名為 `load` 和 `safe_load` 的函數？」

* 「當我寫一個 Python shelf 時，後綴為 `dat` 和 `dir` 的檔案是什麼？」

* 「除了鍵值儲存之外，還有什麼其他類型的資料庫？」

* 「當我讀取檔案時，二進位模式和文字模式有什麼區別？」

* 「bytes 物件和字串有什麼區別？」

* 「什麼是雜湊函數？」

* 「什麼是 MD5 摘要？」

一如往常，如果你在以下任何練習中遇到困難，請考慮向虛擬助理尋求幫助。與你的問題一起，你可能想要貼上本章中相關的函數。

### 練習

寫一個名為 `replace_all` 的函數，它接受模式字串、替換字串和兩個檔案名稱作為參數。
它應該讀取第一個檔案並將內容寫入第二個檔案（如果需要的話創建它）。
如果模式字串出現在內容中的任何地方，它應該被替換為替換字串。

以下是函數的大綱，讓你開始。

In [68]:
def replace_all(old, new, source_path, dest_path):
    # read the contents of the source file
    reader = open(source_path)

    # replace the old string with the new
    
    # write the result into the destination file
    

In [69]:
# Solution

def replace_all(old, new, source_path, dest_path):
    reader = open(source_path)
    contents = reader.read()
    reader.close()
    
    contents = contents.replace(old, new)
    
    writer = open(dest_path, 'w')
    writer.write(contents)
    writer.close()

要測試你的函數，讀取檔案 `photos/notes.txt`，將 `'photos'` 替換為 `'images'`，並將結果寫入檔案 `photos/new_notes.txt`。

In [70]:
source_path = 'photos/notes.txt'
open(source_path).read()

'These photos are from Lorem Picsum at https://picsum.photos\n'

In [71]:
dest_path = 'photos/new_notes.txt'
old = 'photos'
new = 'images'
replace_all(old, new, source_path, dest_path)

In [72]:
open(dest_path).read()

'These images are from Lorem Picsum at https://picsum.images\n'

### 練習

在前面的一節中，我們使用 `shelve` 模組來製作一個從排序的字母字串映射到變位詞列表的鍵值儲存。
要完成範例，寫一個名為 `add_word` 的函數，它接受字串和 shelf 物件作為參數。

它應該對詞的字母進行排序以產生鍵，然後檢查鍵是否已經在 shelf 中。如果沒有，它應該建立一個包含新詞的列表並將其添加到 shelf 中。如果有，它應該將新詞附加到現有值。

In [73]:
# Solution

def add_word(word, db):
    key = sort_word(word)

    if key not in db:
        db[key] = [word]
    else:
        anagrams = db[key]
        anagrams.append(word)
        db[key] = anagrams

你可以使用這個迴圈來測試你的函數。

In [74]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

In [75]:
word_list = open('words.txt').read().split()

db = shelve.open('anagram_map', 'n')
for word in word_list:
    add_word(word, db)

如果一切運作正常，你應該能夠查找像 `'opst'` 這樣的鍵並取得可以用這些字母拼寫的詞列表。

In [76]:
db['opst']

['opts', 'post', 'pots', 'spot', 'stop', 'tops']

In [77]:
for key, value in db.items():
    if len(value) > 8:
        print(value)

['alerts', 'alters', 'artels', 'estral', 'laster', 'ratels', 'salter', 'slater', 'staler', 'stelar', 'talers']
['apers', 'asper', 'pares', 'parse', 'pears', 'prase', 'presa', 'rapes', 'reaps', 'spare', 'spear']
['capers', 'crapes', 'escarp', 'pacers', 'parsec', 'recaps', 'scrape', 'secpar', 'spacer']
['estrin', 'inerts', 'insert', 'inters', 'niters', 'nitres', 'sinter', 'triens', 'trines']
['least', 'setal', 'slate', 'stale', 'steal', 'stela', 'taels', 'tales', 'teals', 'tesla']


In [78]:
db.close()

### 練習

在大量檔案集合中，可能有同一檔案的多個副本，儲存在不同目錄中或使用不同檔案名稱。
此練習的目標是搜尋重複檔案。
作為範例，我們將使用 `photos` 目錄中的圖片檔案。

它將這樣運作：

* 我們將使用來自遍歷目錄一節的 `walk` 函數來搜尋此目錄中以 `config['extensions']` 中的副檔名結尾的檔案。

* 對於每個檔案，我們將使用來自 md5_digest 一節的 `md5_digest` 來計算內容的摘要。

* 使用 shelf，我們將建立從每個摘要到具有該摘要的路徑列表的映射。

* 最後，我們將搜尋 shelf 中映射到多個檔案的任何摘要。

* 如果我們找到任何，我們將使用 `same_contents` 來確認檔案包含相同的資料。

我建議先寫一些函數，然後我們將把它們組合在一起。

1. 為了識別圖片檔案，寫一個名為 `is_image` 的函數，它接受路徑和檔案副檔名列表，如果路徑以列表中的某個副檔名結尾則返回 `True`。提示：使用 `os.path.splitext`——或要求虛擬助理為你寫這個函數。

In [79]:
# Solution

def is_image(path, extensions):
    """Checks whether the path ends with one of the extensions.
    
    path: string file path
    extensions: list of extensions
    
    >>> is_image('photo.jpg', ['jpg', 'jpeg'])
    True
    >>> is_image('PHOTO.JPG', ['jpg', 'jpeg'])
    True
    >>> is_image('notes.txt', ['jpg', 'jpeg'])
    False
    """
    _, extension = os.path.splitext(path)
    return extension.strip('.').lower() in extensions

你可以使用 `doctest` 來測試你的函數。

In [80]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

run_doctests(is_image)

2. 寫一個名為 `add_path` 的函數，它接受路徑和 shelf 作為參數。它應該使用 `md5_digest` 來計算檔案內容的摘要。然後它應該更新 shelf，要麼創建一個從摘要映射到包含路徑的列表的新項目，要麼如果存在的話將路徑附加到列表中。

In [81]:
# Solution

def add_path(path, db):
    digest = md5_digest(path)
    
    if digest not in db:
        paths = [path]
    else:
        paths = db[digest]
        paths.append(path)
        
    db[digest] = paths

3. 寫一個名為 `walk_images` 的 `walk` 版本，它接受目錄並遍歷目錄及其子目錄中的檔案。對於每個檔案，它應該使用 `is_image` 來檢查它是否為圖片檔案，並使用 `add_path` 來將其添加到 shelf 中。

In [87]:
# Solution

def walk_images(dirname):
    for name in os.listdir(dirname):
        path = os.path.join(dirname, name)

        if os.path.isfile(path):
            if is_image(path, config['extensions']):
                add_path(path, db)
        else:
            walk_images(path)    

當一切都運作時，你可以使用以下程式來創建 shelf、搜尋 `photos` 目錄並將路徑添加到 shelf 中，然後檢查是否有多個檔案具有相同摘要。

In [88]:
db = shelve.open('photos/digests', 'n')
walk_images('photos')

for digest, paths in db.items():
    if len(paths) > 1:
        print(paths)

['photos/mar-2023/photo2.jpg', 'photos/jan-2023/photo1.jpg']


你應該找到一對具有相同摘要的檔案。
使用 `same_contents` 來檢查它們是否包含相同的資料。

In [89]:
# Solution

path1, path2 = ['photos/mar-2023/photo2.jpg', 'photos/jan-2023/photo1.jpg']
same_contents(path1, path2)

True

[Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Code license: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)